```markdown
# Projeto de Análise de Dados e BI — Ativa Solar

**Curso:** Análise de Dados e BI (UNIPÊ)  
**Professor:** Daniel Brandão  

---

## Apresentação e Contexto do Projeto

Este projeto prático visa apoiar a empresa **Ativa Solar**, atuante no setor de energia solar fotovoltaica para clientes corporativos (B2B) e residenciais (B2C), na identificação de oportunidades estratégicas de prospecção e expansão de mercado.

### Recorte Geográfico
A análise abrange uma região interestadual contígua composta por **12 municípios** nos estados de Pernambuco (PE) e Paraíba (PB):

*   **Pernambuco (PE):** Abreu e Lima, Goiana, Igarassu, Ilha de Itamaracá, Itambé e Itapissuma.
*   **Paraíba (PB):** Alhandra, Caaporã, Conde, João Pessoa, Pedras de Fogo e Pitimbu.

### Fontes de Dados Utilizadas
1.  **INPE (Instituto Nacional de Pesquisas Espaciais):** Dados de irradiação solar global horizontal (GHI) e irradiação no plano inclinado a 10° (GTI).
2.  **ANEEL (Agência Nacional de Energia Elétrica):** Estatísticas de Micro e Minigeração Distribuída (MMGD) fotovoltaica.
3.  **IBGE (Instituto Brasileiro de Geografia e Estatística):** Estimativas populacionais para o ano de 2026.
4.  **Base Integrada:** Um consolidado que unifica os dados das três fontes acima.

> ⚠️ **Nota Importante:** Este projeto não conta com dados históricos de vendas ou clientes reais da Ativa Solar. Portanto, as informações da ANEEL retratam o panorama geral do mercado de geração distribuída na região, e não o histórico comercial da empresa.
```

```markdown
## Objetivo do Notebook

Este notebook implementa e documenta a fase **L — Limpeza de Dados** do framework de ciência de dados adotado, dividida em quatro etapas iniciais fundamentais:

1.  **Etapa 1:** Identificar e preservar a integridade das bases de dados originais.
2.  **Etapa 2:** Realizar a leitura correta dos arquivos CSV respeitando padrões de codificação e localidade.
3.  **Etapa 3:** Conferir e padronizar as informações geográficas (municípios e códigos IBGE).
4.  **Etapa 4:** Ajustar e validar a tipagem dos dados e as unidades de medida das variáveis.

Por fim, exportaremos as versões tratadas em uma estrutura de pastas organizada e protegida.
```

Configuração Inicial do Ambiente

Este bloco de código importa as bibliotecas fundamentais necessárias para a
manipulação de arquivos (Path, ZipFile) e processamento de dados (pandas).


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
from google.colab import files

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


In [ ]:
nomes_arquivos = {
    "INPE": "INPE_irrad_12_municipios_PE_PB.csv",
    "ANEEL": "ANEEL_MMGD_fotovoltaica_12_municipios_PE_PB_2026-11-10.csv",
    "IBGE": "IBGE_populacao_municipal_12_municipios_PE_PB_2026.csv",
    "Integrada": "base_integrada.csv",
}

arquivos_faltando = []

for nome_base, nome_arquivo in nomes_arquivos.items():
    caminho = Path(nome_arquivo)

    if caminho.exists():
        tamanho_kb = caminho.stat().st_size / 1024
        print(f"✅ {nome_base}: {nome_arquivo} — {tamanho_kb:.2f} KB")
    else:
        arquivos_faltando.append(nome_arquivo)
        print(f"❌ {nome_base}: arquivo não encontrado")

if arquivos_faltando:
    raise FileNotFoundError(
        "Envie os arquivos faltantes antes de continuar: "
        + ", ".join(arquivos_faltando)
    )

print("\nTodos os arquivos necessários foram encontrados.")

✅ INPE: INPE_irrad_12_municipios_PE_PB.csv — 2.76 KB
✅ ANEEL: ANEEL_MMGD_fotovoltaica_12_municipios_PE_PB_2026-11-10.csv — 0.43 KB
✅ IBGE: IBGE_populacao_municipal_12_municipios_PE_PB_2026.csv — 0.39 KB
✅ Integrada: base_integrada.csv — 3.51 KB

Todos os arquivos necessários foram encontrados.


In [ ]:
bases = {}

for nome_base, nome_arquivo in nomes_arquivos.items():
    tabela = pd.read_csv(
        nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        dtype={"codigo_ibge": "string"},
    )

    bases[nome_base] = tabela

    print(f"\n{'=' * 50}")
    print(f"BASE: {nome_base}")
    print(f"Linhas: {len(tabela)}")
    print(f"Colunas: {len(tabela.columns)}")
    print("\nNomes das colunas:")
    print(tabela.columns.tolist())

    print("\nPrimeiras linhas:")
    display(tabela.head())


BASE: INPE
Linhas: 12
Colunas: 33

Nomes das colunas:
['uf', 'codigo_ibge', 'municipio', 'ghi_longitude', 'ghi_latitude', 'ghi_annual_Wh_m2_day', 'ghi_jan_Wh_m2_day', 'ghi_feb_Wh_m2_day', 'ghi_mar_Wh_m2_day', 'ghi_apr_Wh_m2_day', 'ghi_may_Wh_m2_day', 'ghi_jun_Wh_m2_day', 'ghi_jul_Wh_m2_day', 'ghi_aug_Wh_m2_day', 'ghi_sep_Wh_m2_day', 'ghi_oct_Wh_m2_day', 'ghi_nov_Wh_m2_day', 'ghi_dec_Wh_m2_day', 'gti10_longitude', 'gti10_latitude', 'gti10_annual_Wh_m2_day', 'gti10_jan_Wh_m2_day', 'gti10_feb_Wh_m2_day', 'gti10_mar_Wh_m2_day', 'gti10_apr_Wh_m2_day', 'gti10_may_Wh_m2_day', 'gti10_jun_Wh_m2_day', 'gti10_jul_Wh_m2_day', 'gti10_aug_Wh_m2_day', 'gti10_sep_Wh_m2_day', 'gti10_oct_Wh_m2_day', 'gti10_nov_Wh_m2_day', 'gti10_dec_Wh_m2_day']

Primeiras linhas:


,uf,codigo_ibge,municipio,ghi_longitude,ghi_latitude,ghi_annual_Wh_m2_day,ghi_jan_Wh_m2_day,ghi_feb_Wh_m2_day,ghi_mar_Wh_m2_day,ghi_apr_Wh_m2_day,...,gti10_mar_Wh_m2_day,gti10_apr_Wh_m2_day,gti10_may_Wh_m2_day,gti10_jun_Wh_m2_day,gti10_jul_Wh_m2_day,gti10_aug_Wh_m2_day,gti10_sep_Wh_m2_day,gti10_oct_Wh_m2_day,gti10_nov_Wh_m2_day,gti10_dec_Wh_m2_day
0,PB,2500601,Alhandra,-34.9060,-7.4302,5416,5748,5984,6002,5293,...,5998,5445,4950,4563,4587,5338,5593,5759,5765,5727
1,PB,2503001,Caaporã,-34.9058,-7.5139,5410,5746,6018,6004,5306,...,6000,5461,4898,4538,4586,5307,5590,5752,5777,5720
2,PB,2504603,Conde,-34.9002,-7.2579,5449,5812,5992,6019,5313,...,6014,5464,4982,4592,4634,5386,5633,5800,5810,5728
3,PB,2507507,João Pessoa,-34.8647,-7.1159,5530,5862,6002,6009,5365,...,6004,5511,5068,4627,4722,5452,5798,5934,5990,5833
4,PB,2511202,Pedras de Fogo,-35.1125,-7.4025,5403,5740,5940,5925,5335,...,5921,5489,4871,4512,4606,5250,5632,5730,5875,5710



BASE: ANEEL
Linhas: 12
Colunas: 5

Nomes das colunas:
['uf', 'codigo_ibge', 'gd_fv_instalacoes', 'gd_fv_potencia_instalada_kw', 'gd_fv_ucs_com_credito']

Primeiras linhas:


,uf,codigo_ibge,gd_fv_instalacoes,gd_fv_potencia_instalada_kw,gd_fv_ucs_com_credito
0,PB,2500601,122,2876.87,299
1,PB,2503001,129,1213.37,191
2,PB,2504603,898,11078.60,1684
3,PB,2507507,10448,118223.30,16630
4,PB,2511202,295,4731.62,493



BASE: IBGE
Linhas: 12
Colunas: 4

Nomes das colunas:
['uf', 'codigo_ibge', 'municipio', 'populacao_estimada_2026']

Primeiras linhas:


,uf,codigo_ibge,municipio,populacao_estimada_2026
0,PB,2500601,Alhandra,23318
1,PB,2503001,Caaporã,22014
2,PB,2504603,Conde,30446
3,PB,2507507,João Pessoa,906093
4,PB,2511202,Pedras de Fogo,31549



BASE: Integrada
Linhas: 12
Colunas: 44

Nomes das colunas:
['uf', 'codigo_ibge', 'municipio', 'populacao_estimada_2026', 'ghi_longitude', 'ghi_latitude', 'ghi_annual_Wh_m2_day', 'ghi_jan_Wh_m2_day', 'ghi_feb_Wh_m2_day', 'ghi_mar_Wh_m2_day', 'ghi_apr_Wh_m2_day', 'ghi_may_Wh_m2_day', 'ghi_jun_Wh_m2_day', 'ghi_jul_Wh_m2_day', 'ghi_aug_Wh_m2_day', 'ghi_sep_Wh_m2_day', 'ghi_oct_Wh_m2_day', 'ghi_nov_Wh_m2_day', 'ghi_dec_Wh_m2_day', 'gti10_longitude', 'gti10_latitude', 'gti10_annual_Wh_m2_day', 'gti10_jan_Wh_m2_day', 'gti10_feb_Wh_m2_day', 'gti10_mar_Wh_m2_day', 'gti10_apr_Wh_m2_day', 'gti10_may_Wh_m2_day', 'gti10_jun_Wh_m2_day', 'gti10_jul_Wh_m2_day', 'gti10_aug_Wh_m2_day', 'gti10_sep_Wh_m2_day', 'gti10_oct_Wh_m2_day', 'gti10_nov_Wh_m2_day', 'gti10_dec_Wh_m2_day', 'gd_fv_instalacoes', 'gd_fv_potencia_instalada_kw', 'gd_fv_ucs_com_credito', 'gd_fv_kw_por_1000_hab', 'gd_fv_instalacoes_por_1000_hab', 'domicilios_ocupados_2022', 'estabelecimentos_ativos_cnae_selecionados', 'atendimento_confirma

,uf,codigo_ibge,municipio,populacao_estimada_2026,ghi_longitude,ghi_latitude,ghi_annual_Wh_m2_day,ghi_jan_Wh_m2_day,ghi_feb_Wh_m2_day,ghi_mar_Wh_m2_day,...,gd_fv_instalacoes,gd_fv_potencia_instalada_kw,gd_fv_ucs_com_credito,gd_fv_kw_por_1000_hab,gd_fv_instalacoes_por_1000_hab,domicilios_ocupados_2022,estabelecimentos_ativos_cnae_selecionados,atendimento_confirmado,ano_referencia_domicilios,data_referencia_cnpj
0,PB,2500601,Alhandra,23318,-34.9060,-7.4302,5416,5748,5984,6002,...,122,2876.87,299,123.38,5.23,NaN,NaN,NaN,NaN,NaN
1,PB,2503001,Caaporã,22014,-34.9058,-7.5139,5410,5746,6018,6004,...,129,1213.37,191,55.12,5.86,NaN,NaN,NaN,NaN,NaN
2,PB,2504603,Conde,30446,-34.9002,-7.2579,5449,5812,5992,6019,...,898,11078.60,1684,363.88,29.49,NaN,NaN,NaN,NaN,NaN
3,PB,2507507,João Pessoa,906093,-34.8647,-7.1159,5530,5862,6002,6009,...,10448,118223.30,16630,130.48,11.53,NaN,NaN,NaN,NaN,NaN
4,PB,2511202,Pedras de Fogo,31549,-35.1125,-7.4025,5403,5740,5940,5925,...,295,4731.62,493,149.98,9.35,NaN,NaN,NaN,NaN,NaN


In [ ]:
municipios_referencia = {
    "2500601": ("PB", "Alhandra"),
    "2503001": ("PB", "Caaporã"),
    "2504603": ("PB", "Conde"),
    "2507507": ("PB", "João Pessoa"),
    "2511202": ("PB", "Pedras de Fogo"),
    "2511905": ("PB", "Pitimbu"),
    "2600054": ("PE", "Abreu e Lima"),
    "2606200": ("PE", "Goiana"),
    "2606804": ("PE", "Igarassu"),
    "2607604": ("PE", "Ilha de Itamaracá"),
    "2607653": ("PE", "Itambé"),
    "2607752": ("PE", "Itapissuma"),
}

referencia = pd.DataFrame(
    [
        {
            "codigo_ibge": codigo,
            "uf": dados[0],
            "municipio": dados[1],
        }
        for codigo, dados in municipios_referencia.items()
    ]
)

print("Municípios definidos para o projeto:")
display(referencia)

Municípios definidos para o projeto:


,codigo_ibge,uf,municipio
0,2500601,PB,Alhandra
1,2503001,PB,Caaporã
2,2504603,PB,Conde
3,2507507,PB,João Pessoa
4,2511202,PB,Pedras de Fogo
5,2511905,PB,Pitimbu
6,2600054,PE,Abreu e Lima
7,2606200,PE,Goiana
8,2606804,PE,Igarassu
9,2607604,PE,Ilha de Itamaracá


In [ ]:
codigos_esperados = set(municipios_referencia)
problemas_geograficos = []

for nome_base, tabela in bases.items():
    print(f"\n{'=' * 50}")
    print(f"Conferindo a base: {nome_base}")

    if "codigo_ibge" not in tabela.columns:
        raise ValueError(
            f"A base {nome_base} não possui a coluna codigo_ibge."
        )

    # Remove espaços antes e depois dos códigos.
    tabela["codigo_ibge"] = (
        tabela["codigo_ibge"]
        .astype("string")
        .str.strip()
    )

    # Remove espaços externos dos campos de texto, quando existentes.
    for coluna in ["uf", "municipio"]:
        if coluna in tabela.columns:
            tabela[coluna] = (
                tabela[coluna]
                .astype("string")
                .str.strip()
            )

    formato_valido = tabela["codigo_ibge"].str.fullmatch(
        r"\d{7}",
        na=False,
    )

    if not formato_valido.all():
        invalidos = tabela.loc[
            ~formato_valido,
            "codigo_ibge",
        ].tolist()

        problemas_geograficos.append(
            f"{nome_base}: códigos inválidos ou ausentes: {invalidos}"
        )

    duplicados = tabela.loc[
        tabela["codigo_ibge"].duplicated(keep=False),
        "codigo_ibge",
    ].tolist()

    if duplicados:
        problemas_geograficos.append(
            f"{nome_base}: códigos duplicados: {duplicados}"
        )

    codigos_encontrados = set(
        tabela["codigo_ibge"].dropna().tolist()
    )

    faltando = sorted(codigos_esperados - codigos_encontrados)
    extras = sorted(codigos_encontrados - codigos_esperados)

    if faltando:
        problemas_geograficos.append(
            f"{nome_base}: códigos esperados ausentes: {faltando}"
        )

    if extras:
        problemas_geograficos.append(
            f"{nome_base}: códigos fora do recorte: {extras}"
        )

    for _, linha in tabela.iterrows():
        codigo = linha["codigo_ibge"]

        if pd.isna(codigo) or codigo not in municipios_referencia:
            continue

        uf_esperada, nome_esperado = municipios_referencia[codigo]

        if "uf" in tabela.columns:
            uf_atual = linha["uf"]

            if pd.isna(uf_atual) or uf_atual != uf_esperada:
                problemas_geograficos.append(
                    f"{nome_base}: UF divergente para {codigo}: "
                    f"{uf_atual!r}; esperado {uf_esperada!r}"
                )

        if "municipio" in tabela.columns:
            nome_atual = linha["municipio"]

            if pd.isna(nome_atual) or nome_atual != nome_esperado:
                problemas_geograficos.append(
                    f"{nome_base}: nome divergente para {codigo}: "
                    f"{nome_atual!r}; esperado {nome_esperado!r}"
                )

    print(f"Quantidade de linhas: {len(tabela)}")
    print(f"Códigos únicos: {tabela['codigo_ibge'].nunique()}")
    print(f"Códigos faltando: {faltando}")
    print(f"Códigos extras: {extras}")
    print(f"Códigos duplicados: {duplicados}")

if problemas_geograficos:
    print("\n⚠️ Divergências encontradas:")

    for problema in problemas_geograficos:
        print("-", problema)

    raise ValueError(
        "Investiguem as divergências antes de continuar."
    )

print("\n✅ Municípios e códigos conferidos nas quatro bases.")


Conferindo a base: INPE
Quantidade de linhas: 12
Códigos únicos: 12
Códigos faltando: []
Códigos extras: []
Códigos duplicados: []

Conferindo a base: ANEEL
Quantidade de linhas: 12
Códigos únicos: 12
Códigos faltando: []
Códigos extras: []
Códigos duplicados: []

Conferindo a base: IBGE
Quantidade de linhas: 12
Códigos únicos: 12
Códigos faltando: []
Códigos extras: []
Códigos duplicados: []

Conferindo a base: Integrada
Quantidade de linhas: 12
Códigos únicos: 12
Códigos faltando: []
Códigos extras: []
Códigos duplicados: []

✅ Municípios e códigos conferidos nas quatro bases.


In [ ]:
colunas_inteiras = {
    "ANEEL": [
        "gd_fv_instalacoes",
        "gd_fv_ucs_com_credito",
    ],
    "IBGE": [
        "populacao_estimada_2026",
    ],
    "Integrada": [
        "populacao_estimada_2026",
        "gd_fv_instalacoes",
        "gd_fv_ucs_com_credito",
    ],
}

colunas_decimais = {
    "ANEEL": [
        "gd_fv_potencia_instalada_kw",
    ],
    "Integrada": [
        "gd_fv_potencia_instalada_kw",
        "gd_fv_kw_por_1000_hab",
        "gd_fv_instalacoes_por_1000_hab",
    ],
}

problemas_tipos = []


def converter_coluna(tabela, nome_base, coluna, inteira=False):
    """Converte a coluna e registra valores que precisam de revisão."""

    if coluna not in tabela.columns:
        problemas_tipos.append(
            f"{nome_base}: coluna não encontrada: {coluna}"
        )
        return

    originais = tabela[coluna].copy()

    convertidos = pd.to_numeric(
        originais,
        errors="coerce",
    )

    # Identifica valores preenchidos que não viraram números.
    falhas = originais.notna() & convertidos.isna()

    if falhas.any():
        problemas_tipos.append(
            f"{nome_base}.{coluna}: valores não numéricos: "
            f"{originais[falhas].tolist()}"
        )

    if inteira:
        fracionarios = convertidos.notna() & (
            convertidos % 1 != 0
        )

        if fracionarios.any():
            problemas_tipos.append(
                f"{nome_base}.{coluna}: valores não inteiros: "
                f"{convertidos[fracionarios].tolist()}"
            )
            return

        tabela[coluna] = convertidos.astype("Int64")

    else:
        tabela[coluna] = convertidos


# Converter campos inteiros.
for nome_base, colunas in colunas_inteiras.items():
    for coluna in colunas:
        converter_coluna(
            bases[nome_base],
            nome_base,
            coluna,
            inteira=True,
        )

# Converter campos decimais.
for nome_base, colunas in colunas_decimais.items():
    for coluna in colunas:
        converter_coluna(
            bases[nome_base],
            nome_base,
            coluna,
        )

# Converter coordenadas e campos solares.
for nome_base in ["INPE", "Integrada"]:
    tabela = bases[nome_base]

    colunas_solares = [
        coluna
        for coluna in tabela.columns
        if coluna.startswith(("ghi_", "gti10_"))
    ]

    if not colunas_solares:
        problemas_tipos.append(
            f"{nome_base}: nenhum campo GHI ou GTI encontrado."
        )

    for coluna in colunas_solares:
        converter_coluna(
            tabela,
            nome_base,
            coluna,
        )

if problemas_tipos:
    print("⚠️ Problemas encontrados:")

    for problema in problemas_tipos:
        print("-", problema)

    raise ValueError(
        "Investiguem os problemas de tipo antes de exportar."
    )

print("✅ Conversão dos campos concluída.")

✅ Conversão dos campos concluída.


In [ ]:
for nome_base, tabela in bases.items():
    print(f"\n{'=' * 50}")
    print(f"TIPOS DE DADOS — {nome_base}")
    print(tabela.dtypes)


TIPOS DE DADOS — INPE
uf                        string[python]
codigo_ibge               string[python]
municipio                 string[python]
ghi_longitude                    float64
ghi_latitude                     float64
ghi_annual_Wh_m2_day               int64
ghi_jan_Wh_m2_day                  int64
ghi_feb_Wh_m2_day                  int64
ghi_mar_Wh_m2_day                  int64
ghi_apr_Wh_m2_day                  int64
ghi_may_Wh_m2_day                  int64
ghi_jun_Wh_m2_day                  int64
ghi_jul_Wh_m2_day                  int64
ghi_aug_Wh_m2_day                  int64
ghi_sep_Wh_m2_day                  int64
ghi_oct_Wh_m2_day                  int64
ghi_nov_Wh_m2_day                  int64
ghi_dec_Wh_m2_day                  int64
gti10_longitude                  float64
gti10_latitude                   float64
gti10_annual_Wh_m2_day             int64
gti10_jan_Wh_m2_day                int64
gti10_feb_Wh_m2_day                int64
gti10_mar_Wh_m2_day               

In [ ]:
pasta_saida = Path("bases_tratadas_etapas_01_a_04")
pasta_saida.mkdir(exist_ok=True)

for nome_base, tabela in bases.items():
    nome_saida = f"{nome_base.lower()}_etapas_01_a_04.csv"
    caminho_saida = pasta_saida / nome_saida

    tabela.to_csv(
        caminho_saida,
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False,
    )

    print(f"✅ Arquivo salvo: {caminho_saida}")

✅ Arquivo salvo: bases_tratadas_etapas_01_a_04/inpe_etapas_01_a_04.csv
✅ Arquivo salvo: bases_tratadas_etapas_01_a_04/aneel_etapas_01_a_04.csv
✅ Arquivo salvo: bases_tratadas_etapas_01_a_04/ibge_etapas_01_a_04.csv
✅ Arquivo salvo: bases_tratadas_etapas_01_a_04/integrada_etapas_01_a_04.csv


In [ ]:
nome_zip = "Bases_AtivaSolar_Etapas_01_a_04.zip"

with ZipFile(nome_zip, "w", ZIP_DEFLATED) as arquivo_zip:
    for arquivo in sorted(pasta_saida.glob("*.csv")):
        arquivo_zip.write(
            arquivo,
            arcname=arquivo.name,
        )

print(f"✅ ZIP criado: {nome_zip}")

files.download(nome_zip)

✅ ZIP criado: Bases_AtivaSolar_Etapas_01_a_04.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Etapa 5 — Validar granularidade e duplicidades

A checagem confirma que cada uma das quatro tabelas representa uma linha por município. Duplicidades, códigos inválidos, faltantes ou fora do recorte são exibidos para investigação; nenhuma linha é removida ou agregada automaticamente.

In [ ]:
auditoria_granularidade_registros = []
problemas_granularidade = []
codigos_esperados = set(municipios_referencia.keys())


def _exibir_etapa(tabela):
    funcao_display = globals().get("display")
    if callable(funcao_display):
        funcao_display(tabela)
    else:
        print(tabela.to_string(index=False))


for nome_base, tabela in bases.items():
    if "codigo_ibge" not in tabela.columns:
        problemas_granularidade.append(
            f"{nome_base}: coluna codigo_ibge não encontrada."
        )
        auditoria_granularidade_registros.append({
            "base": nome_base,
            "linhas": len(tabela),
            "codigos_unicos": 0,
            "linhas_duplicadas": 0,
            "codigos_ausentes": len(codigos_esperados),
            "codigos_extras": 0,
        })
        continue

    codigos = tabela["codigo_ibge"].astype("string").str.strip()
    mascara_duplicados = codigos.duplicated(keep=False)
    linhas_duplicadas = tabela.loc[mascara_duplicados].copy()
    codigos_invalidos = codigos.loc[
        ~codigos.str.fullmatch(r"\d{7}", na=False)
    ].tolist()
    encontrados = set(codigos.dropna().tolist())
    faltantes = sorted(codigos_esperados - encontrados)
    extras = sorted(encontrados - codigos_esperados)

    auditoria_granularidade_registros.append({
        "base": nome_base,
        "linhas": len(tabela),
        "codigos_unicos": int(codigos.nunique(dropna=True)),
        "linhas_duplicadas": int(mascara_duplicados.sum()),
        "codigos_ausentes": len(faltantes),
        "codigos_extras": len(extras),
    })

    print(
        f"{nome_base}: linhas={len(tabela)}, "
        f"municípios={codigos.nunique(dropna=True)}, "
        f"linhas duplicadas={int(mascara_duplicados.sum())}"
    )

    if not linhas_duplicadas.empty:
        print(f"Linhas conflitantes em {nome_base}:")
        _exibir_etapa(linhas_duplicadas.sort_values("codigo_ibge"))
        problemas_granularidade.append(
            f"{nome_base}: códigos municipais duplicados."
        )

    if codigos_invalidos:
        problemas_granularidade.append(
            f"{nome_base}: códigos inválidos ou ausentes: {codigos_invalidos}"
        )
    if faltantes:
        problemas_granularidade.append(
            f"{nome_base}: códigos esperados ausentes: {faltantes}"
        )
    if extras:
        problemas_granularidade.append(
            f"{nome_base}: códigos fora do recorte: {extras}"
        )

auditoria_granularidade = pd.DataFrame(auditoria_granularidade_registros)
_exibir_etapa(auditoria_granularidade)

if problemas_granularidade:
    for problema in problemas_granularidade:
        print(f"⚠️ {problema}")
    raise ValueError(
        "A etapa 5 encontrou problemas de granularidade. "
        "Investigue as fontes antes de continuar."
    )

print("Etapa 5 concluída: 12 municípios, uma linha por código em cada base.")

## Etapa 6 — Tratar ausências e validar valores

A auditoria separa os campos pendentes de valores ausentes inesperados e verifica tipos, faixas e coerência das unidades. Nenhum campo será preenchido por zero ou por estimativa; correções dependem de confirmação na fonte.

In [ ]:
import math

campos_pendentes = {
    "domicilios_ocupados_2022": "não coletado",
    "estabelecimentos_ativos_cnae_selecionados": "não coletado",
    "atendimento_confirmado": "pendente de confirmação",
    "ano_referencia_domicilios": "não coletado",
    "data_referencia_cnpj": "não coletado",
}
meses_inpe = [
    "jan", "feb", "mar", "apr", "may", "jun",
    "jul", "aug", "sep", "oct", "nov", "dec",
]
auditoria_valores_registros = []
status_pendencias_registros = []
problemas_valores = []


def _registrar_problema_valor(base, coluna, motivo, valores=None):
    detalhe = f"{base}.{coluna}: {motivo}"
    if valores is not None:
        detalhe += f" — {valores}"
    problemas_valores.append(detalhe)


for nome_base, tabela in bases.items():
    if nome_base in ("INPE", "Integrada"):
        colunas_numericas = [
            coluna for coluna in tabela.columns
            if coluna.startswith(("ghi_", "gti10_"))
        ]
        if nome_base == "Integrada":
            colunas_numericas.extend(
                coluna for coluna in [
                    "populacao_estimada_2026",
                    "gd_fv_instalacoes",
                    "gd_fv_potencia_instalada_kw",
                    "gd_fv_ucs_com_credito",
                    "gd_fv_kw_por_1000_hab",
                    "gd_fv_instalacoes_por_1000_hab",
                ]
                if coluna in tabela.columns
            )
    elif nome_base == "ANEEL":
        colunas_numericas = [
            "gd_fv_instalacoes",
            "gd_fv_potencia_instalada_kw",
            "gd_fv_ucs_com_credito",
        ]
    elif nome_base == "IBGE":
        colunas_numericas = ["populacao_estimada_2026"]
    else:
        colunas_numericas = []

    for coluna in colunas_numericas:
        if coluna not in tabela.columns:
            _registrar_problema_valor(
                nome_base, coluna, "coluna numérica obrigatória não encontrada"
            )
            continue

        original = tabela[coluna]
        numerico = pd.to_numeric(original, errors="coerce")
        nao_numericos = original.notna() & numerico.isna()
        nao_finitos = numerico.map(
            lambda valor: False
            if pd.isna(valor)
            else not math.isfinite(float(valor))
        )
        ausentes = int(original.isna().sum())
        min_valor = numerico.min(skipna=True)
        max_valor = numerico.max(skipna=True)
        situacao = "válido"

        if ausentes:
            situacao = "revisar"
            _registrar_problema_valor(
                nome_base, coluna, f"{ausentes} ausência(s) inesperada(s)"
            )
        if nao_numericos.any():
            situacao = "revisar"
            _registrar_problema_valor(
                nome_base,
                coluna,
                "valor(es) preenchido(s) não numérico(s)",
                original.loc[nao_numericos].tolist(),
            )
        if nao_finitos.any():
            situacao = "revisar"
            _registrar_problema_valor(
                nome_base,
                coluna,
                "valor(es) infinito(s) ou não finito(s)",
                original.loc[nao_finitos].tolist(),
            )

        if coluna == "populacao_estimada_2026":
            invalidos = numerico.notna() & (numerico <= 0)
        elif coluna.startswith(("ghi_", "gti10_")) and not coluna.endswith(
            ("longitude", "latitude")
        ):
            invalidos = numerico.notna() & (numerico < 0)
        elif coluna.startswith("gd_fv_"):
            invalidos = numerico.notna() & (numerico < 0)
        else:
            invalidos = pd.Series(False, index=tabela.index)

        if invalidos.any():
            situacao = "revisar"
            _registrar_problema_valor(
                nome_base,
                coluna,
                "valor(es) negativo(s) ou fora da faixa documentada",
                original.loc[invalidos].tolist(),
            )

        if coluna in (
            "populacao_estimada_2026",
            "gd_fv_instalacoes",
            "gd_fv_ucs_com_credito",
        ):
            fracionarios = numerico.notna() & (numerico % 1 != 0)
            if fracionarios.any():
                situacao = "revisar"
                _registrar_problema_valor(
                    nome_base,
                    coluna,
                    "contagem com valor fracionário",
                    original.loc[fracionarios].tolist(),
                )

        auditoria_valores_registros.append({
            "base": nome_base,
            "coluna": coluna,
            "ausentes": ausentes,
            "não_numéricos": int(nao_numericos.sum()),
            "não_finitos": int(nao_finitos.sum()),
            "mínimo": min_valor,
            "máximo": max_valor,
            "situação": situacao,
        })


for nome_base in ("INPE", "Integrada"):
    tabela = bases[nome_base]
    for prefixo in ("ghi", "gti10"):
        colunas_mes = [f"{prefixo}_{mes}_Wh_m2_day" for mes in meses_inpe]
        faltantes_mes = [coluna for coluna in colunas_mes if coluna not in tabela]
        if faltantes_mes:
            _registrar_problema_valor(
                nome_base, prefixo, "meses esperados ausentes", faltantes_mes
            )

    for prefixo in ("ghi", "gti10"):
        longitude = f"{prefixo}_longitude"
        latitude = f"{prefixo}_latitude"
        for coluna in (longitude, latitude):
            if coluna not in tabela:
                _registrar_problema_valor(
                    nome_base, coluna, "coordenada obrigatória não encontrada"
                )
        if longitude in tabela and latitude in tabela:
            lon = pd.to_numeric(tabela[longitude], errors="coerce")
            lat = pd.to_numeric(tabela[latitude], errors="coerce")
            # Envelope geográfico amplo para conferir sinal e plausibilidade em PE/PB.
            fora_regiao = (
                ~lon.between(-40, -33)
                | ~lat.between(-10, -5)
            )
            if fora_regiao.any():
                _registrar_problema_valor(
                    nome_base,
                    f"{prefixo}_longitude/latitude",
                    "coordenada fora do envelope regional de conferência",
                    tabela.loc[fora_regiao, [longitude, latitude]].to_dict("records"),
                )


tabela_integrada_origem = bases["Integrada"]
for coluna, classificacao in campos_pendentes.items():
    if coluna not in tabela_integrada_origem.columns:
        _registrar_problema_valor(
            "Integrada", coluna, "campo pendente esperado não encontrado"
        )
        ausentes = len(tabela_integrada_origem)
        situacao = "campo não encontrado"
    else:
        ausentes = int(tabela_integrada_origem[coluna].isna().sum())
        situacao = (
            classificacao
            if ausentes == len(tabela_integrada_origem)
            else "preenchido; conferir fonte e referência"
        )
    status_pendencias_registros.append({
        "campo": coluna,
        "ausentes": ausentes,
        "total_municípios": len(tabela_integrada_origem),
        "classificação": situacao,
    })

auditoria_valores = pd.DataFrame(auditoria_valores_registros)
status_ausencias = pd.DataFrame(status_pendencias_registros)
print("Resumo das validações numéricas; extremos são informativos e devem ser conferidos na fonte.")
_exibir_etapa(auditoria_valores)
print("Ausências conhecidas, mantidas como ausências:")
_exibir_etapa(status_ausencias)

if problemas_valores:
    for problema in problemas_valores:
        print(f"⚠️ {problema}")
    raise ValueError(
        "A etapa 6 encontrou valores que precisam de investigação na fonte. "
        "Nenhuma correção ou imputação automática foi aplicada."
    )

print("Etapa 6 concluída: ausências classificadas e valores numéricos validados.")

## Etapa 7 — Integrar fontes e recalcular indicadores

A integração usa o código IBGE e a referência de 12 municípios. Cada junção é um-para-um e registra correspondências; nomes e UFs das fontes são conferidos antes de manter a identificação oficial. Os dois indicadores por mil habitantes são recalculados após as validações.

In [ ]:
import math

problemas_geograficos_integracao = []
for nome_base in ("INPE", "ANEEL", "IBGE", "Integrada"):
    tabela = bases[nome_base]
    for _, linha in tabela.iterrows():
        codigo = str(linha["codigo_ibge"]).strip()
        if codigo not in municipios_referencia:
            continue
        uf_esperada, municipio_esperado = municipios_referencia[codigo]
        if "uf" in tabela.columns:
            uf_atual = str(linha["uf"]).strip()
            if uf_atual != uf_esperada:
                problemas_geograficos_integracao.append(
                    f"{nome_base}: UF de {codigo} é {uf_atual!r}; "
                    f"esperada {uf_esperada!r}."
                )
        if "municipio" in tabela.columns:
            municipio_atual = str(linha["municipio"]).strip()
            if municipio_atual != municipio_esperado:
                problemas_geograficos_integracao.append(
                    f"{nome_base}: município de {codigo} é "
                    f"{municipio_atual!r}; esperado {municipio_esperado!r}."
                )

if problemas_geograficos_integracao:
    for problema in problemas_geograficos_integracao:
        print(f"⚠️ {problema}")
    raise ValueError(
        "A identificação geográfica diverge da referência municipal. "
        "Investigue as fontes antes de integrar."
    )

referencia_etapa_07 = pd.DataFrame([
    {
        "codigo_ibge": str(codigo),
        "uf": dados[0],
        "municipio": dados[1],
    }
    for codigo, dados in municipios_referencia.items()
])
integrada_nova = referencia_etapa_07.copy()
auditoria_juncoes_registros = []
problemas_juncoes = []

fontes_para_juncao = ("IBGE", "INPE", "ANEEL")
for nome_base in fontes_para_juncao:
    tabela = bases[nome_base]
    colunas_dados = [
        coluna for coluna in tabela.columns
        if coluna not in ("codigo_ibge", "uf", "municipio")
    ]
    sobrepostas = [
        coluna for coluna in colunas_dados
        if coluna in integrada_nova.columns
    ]
    if sobrepostas:
        raise ValueError(
            f"{nome_base}: colunas sobrepostas precisam ser revisadas: "
            f"{sobrepostas}"
        )

    nome_indicador = f"correspondencia_{nome_base.lower()}"
    integrada_nova = integrada_nova.merge(
        tabela[["codigo_ibge"] + colunas_dados],
        on="codigo_ibge",
        how="left",
        validate="one_to_one",
        indicator=nome_indicador,
    )
    sem_correspondencia = integrada_nova.loc[
        integrada_nova[nome_indicador] != "both", "codigo_ibge"
    ].tolist()
    auditoria_juncoes_registros.append({
        "fonte": nome_base,
        "linhas_referencia": len(integrada_nova),
        "correspondencias": int((integrada_nova[nome_indicador] == "both").sum()),
        "sem_correspondencia": len(sem_correspondencia),
    })
    if sem_correspondencia:
        problemas_juncoes.append(
            f"{nome_base}: códigos sem correspondência: {sem_correspondencia}"
        )
    integrada_nova = integrada_nova.drop(columns=nome_indicador)

colunas_pendentes = list(campos_pendentes)
tabela_pendencias = bases["Integrada"][["codigo_ibge"] + colunas_pendentes]
integrada_nova = integrada_nova.merge(
    tabela_pendencias,
    on="codigo_ibge",
    how="left",
    validate="one_to_one",
    indicator="correspondencia_campos_pendentes",
)
sem_pendencias = integrada_nova.loc[
    integrada_nova["correspondencia_campos_pendentes"] != "both",
    "codigo_ibge",
].tolist()
auditoria_juncoes_registros.append({
    "fonte": "campos pendentes da base integrada atual",
    "linhas_referencia": len(integrada_nova),
    "correspondencias": int(
        (integrada_nova["correspondencia_campos_pendentes"] == "both").sum()
    ),
    "sem_correspondencia": len(sem_pendencias),
})
if sem_pendencias:
    problemas_juncoes.append(
        f"Campos pendentes sem correspondência: {sem_pendencias}"
    )
integrada_nova = integrada_nova.drop(
    columns="correspondencia_campos_pendentes"
)

if len(integrada_nova) != len(municipios_referencia):
    problemas_juncoes.append(
        f"Linhas após junções={len(integrada_nova)}; "
        f"esperadas={len(municipios_referencia)}."
    )
if integrada_nova["codigo_ibge"].nunique() != len(municipios_referencia):
    problemas_juncoes.append(
        "A base integrada não possui um código IBGE único por município."
    )

auditoria_juncoes = pd.DataFrame(auditoria_juncoes_registros)
_exibir_etapa(auditoria_juncoes)
if problemas_juncoes:
    for problema in problemas_juncoes:
        print(f"⚠️ {problema}")
    raise ValueError(
        "A etapa 7 encontrou municípios sem correspondência. "
        "Nenhum município foi excluído."
    )

populacao = pd.to_numeric(
    integrada_nova["populacao_estimada_2026"], errors="coerce"
)
instalacoes = pd.to_numeric(
    integrada_nova["gd_fv_instalacoes"], errors="coerce"
)
potencia_kw = pd.to_numeric(
    integrada_nova["gd_fv_potencia_instalada_kw"], errors="coerce"
)
if populacao.isna().any() or (populacao <= 0).any():
    raise ValueError("A população precisa estar preenchida e ser maior que zero.")
if instalacoes.isna().any() or (instalacoes < 0).any():
    raise ValueError("As instalações ANEEL precisam estar preenchidas e ser >= 0.")
if potencia_kw.isna().any() or (potencia_kw < 0).any():
    raise ValueError("A potência ANEEL precisa estar preenchida e ser >= 0.")

integrada_nova["gd_fv_kw_por_1000_hab"] = (
    potencia_kw / populacao * 1000
).round(2)
integrada_nova["gd_fv_instalacoes_por_1000_hab"] = (
    instalacoes / populacao * 1000
).round(2)

comparacao_indicadores_registros = []
divergencias_indicadores_registros = []
for indicador in (
    "gd_fv_kw_por_1000_hab",
    "gd_fv_instalacoes_por_1000_hab",
):
    antigos = bases["Integrada"][["codigo_ibge", indicador]].rename(
        columns={indicador: "valor_anterior"}
    )
    comparacao = integrada_nova[["codigo_ibge", indicador]].merge(
        antigos,
        on="codigo_ibge",
        how="left",
        validate="one_to_one",
    )
    comparacao["valor_recalculado"] = comparacao[indicador]
    comparacao["diverge_arredondado_2_casas"] = (
        comparacao["valor_anterior"].notna()
        & (
            comparacao["valor_anterior"].round(2)
            != comparacao["valor_recalculado"].round(2)
        )
    )
    divergentes = comparacao.loc[
        comparacao["diverge_arredondado_2_casas"]
    ]
    comparacao_indicadores_registros.append({
        "indicador": indicador,
        "municípios_comparados": int(comparacao["valor_anterior"].notna().sum()),
        "divergências_após_arredondar_2_casas": len(divergentes),
    })
    for _, linha in divergentes.iterrows():
        divergencias_indicadores_registros.append({
            "codigo_ibge": linha["codigo_ibge"],
            "indicador": indicador,
            "valor_anterior": linha["valor_anterior"],
            "valor_recalculado": linha["valor_recalculado"],
        })

comparacao_indicadores = pd.DataFrame(comparacao_indicadores_registros)
divergencias_indicadores = pd.DataFrame(
    divergencias_indicadores_registros,
    columns=[
        "codigo_ibge",
        "indicador",
        "valor_anterior",
        "valor_recalculado",
    ],
)
print("Comparação dos indicadores existentes com os valores recalculados:")
_exibir_etapa(comparacao_indicadores)
if not divergencias_indicadores.empty:
    print("Investigue as diferenças abaixo antes de interpretar os indicadores:")
    _exibir_etapa(divergencias_indicadores)

colunas_base_atual = list(bases["Integrada"].columns)
colunas_integradas_finais = [
    coluna for coluna in colunas_base_atual
    if coluna in integrada_nova.columns
]
colunas_integradas_finais.extend(
    coluna for coluna in integrada_nova.columns
    if coluna not in colunas_integradas_finais
)
integrada_nova = integrada_nova[colunas_integradas_finais]

print(
    f"Etapa 7 concluída: {len(integrada_nova)} municípios integrados "
    "por codigo_ibge; indicadores recalculados."
)

## Etapa 8 — Exportar e documentar a qualidade

A exportação preserva as fontes originais, os códigos IBGE como texto e os campos sem fonte confirmada como ausentes. O relatório de qualidade e o dicionário descrevem referências, unidades, regras aplicadas e limitações. No Colab, os três arquivos finais são reunidos para download.

In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
from zipfile import ZIP_DEFLATED, ZipFile

pasta_saida = Path(
    "Base de dados/AtivaSolar_Bases_Organizadas/05_bases_tratadas"
)
pasta_saida.mkdir(parents=True, exist_ok=True)

caminho_base_limpa = pasta_saida / "base_integrada_limpa.csv"
caminho_relatorio_qualidade = (
    pasta_saida / "relatorio_qualidade_base_integrada.md"
)
caminho_dicionario = pasta_saida / "dicionario_base_integrada.csv"

integrada_nova.to_csv(
    caminho_base_limpa,
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
)

meses_pt = {
    "jan": "janeiro", "feb": "fevereiro", "mar": "março",
    "apr": "abril", "may": "maio", "jun": "junho",
    "jul": "julho", "aug": "agosto", "sep": "setembro",
    "oct": "outubro", "nov": "novembro", "dec": "dezembro",
}
limite_inpe = (
    "Referência histórica na sede municipal, série 1999–2015; "
    "não representa todos os bairros, telhados ou desempenho comercial."
)
metadados_variaveis = {
    "uf": (
        "Sigla da unidade federativa do município.",
        "sigla",
        "Tabela de referência municipal do projeto",
        "Recorte de 12 municípios",
        "Identificação geográfica; não representa cobertura confirmada da empresa.",
    ),
    "codigo_ibge": (
        "Código IBGE do município usado como chave de integração.",
        "código textual de 7 dígitos",
        "IBGE / referência municipal do projeto",
        "Recorte de 12 municípios",
        "Manter como texto para preservar a chave.",
    ),
    "municipio": (
        "Nome oficial do município associado ao código IBGE.",
        "nome",
        "IBGE / referência municipal do projeto",
        "Recorte de 12 municípios",
        "Ilha de Itamaracá mantém a grafia oficial usada no projeto.",
    ),
    "populacao_estimada_2026": (
        "População municipal estimada pelo IBGE.",
        "habitantes",
        "IBGE — Estimativas de População 2026",
        "Referência de 01/07/2026",
        "Não equivale a domicílios, consumidores elegíveis ou clientes potenciais.",
    ),
    "gd_fv_instalacoes": (
        "Quantidade de instalações fotovoltaicas de geração distribuída.",
        "instalações",
        "ANEEL — MMGD fotovoltaica",
        "Fotografia com referência em 10/11/2026",
        "Presença geral de geração distribuída; não mede vendas da Ativa Solar.",
    ),
    "gd_fv_potencia_instalada_kw": (
        "Potência fotovoltaica instalada agregada no município.",
        "kW",
        "ANEEL — MMGD fotovoltaica",
        "Fotografia com referência em 10/11/2026",
        "Presença geral de geração distribuída; não mede vendas da Ativa Solar.",
    ),
    "gd_fv_ucs_com_credito": (
        "Unidades consumidoras que recebem créditos de geração distribuída.",
        "unidades consumidoras",
        "ANEEL — MMGD fotovoltaica",
        "Fotografia com referência em 10/11/2026",
        "Não equivale a clientes únicos nem ao número de instalações.",
    ),
    "gd_fv_kw_por_1000_hab": (
        "Potência FV instalada dividida pela população e multiplicada por 1.000.",
        "kW por 1.000 habitantes",
        "ANEEL 10/11/2026 + IBGE população 2026",
        "ANEEL: 10/11/2026; população: 01/07/2026",
        "Razão descritiva; não representa vendas, clientes ou potencial comprovado.",
    ),
    "gd_fv_instalacoes_por_1000_hab": (
        "Instalações FV divididas pela população e multiplicadas por 1.000.",
        "instalações por 1.000 habitantes",
        "ANEEL 10/11/2026 + IBGE população 2026",
        "ANEEL: 10/11/2026; população: 01/07/2026",
        "Não é percentual de domicílios com energia solar nem vendas da empresa.",
    ),
    "domicilios_ocupados_2022": (
        "Domicílios particulares permanentes ocupados no Censo.",
        "domicílios",
        "IBGE — Censo 2022; coleta pendente",
        "2022; valor ainda não coletado",
        "Vazio significa pendente, não zero; requer conferência das categorias da tabela.",
    ),
    "estabelecimentos_ativos_cnae_selecionados": (
        "Estabelecimentos ativos nos CNAEs que serão acordados com o cliente.",
        "estabelecimentos",
        "Receita Federal — dados abertos do CNPJ; coleta pendente",
        "Data de referência pendente",
        "CNAEs ainda precisam ser definidos; código municipal da Receita exige correspondência ao IBGE.",
    ),
    "atendimento_confirmado": (
        "Indica se a empresa confirmou atendimento ao município.",
        "categoria Sim/Não",
        "Confirmação da Ativa Solar; pendente",
        "Data de confirmação pendente",
        "Vazio significa confirmação pendente, não atendimento negado.",
    ),
    "ano_referencia_domicilios": (
        "Ano de referência da variável de domicílios.",
        "ano",
        "IBGE — Censo 2022; coleta pendente",
        "Pendente",
        "Metadado permanece ausente até a coleta da variável de domicílios.",
    ),
    "data_referencia_cnpj": (
        "Data de referência ou extração do cadastro de estabelecimentos.",
        "data",
        "Receita Federal — CNPJ; coleta pendente",
        "Pendente",
        "Metadado permanece ausente até a coleta dos estabelecimentos.",
    ),
}

registros_dicionario = []
for coluna in integrada_nova.columns:
    if coluna in metadados_variaveis:
        definicao, unidade, fonte, referencia, limitacao = metadados_variaveis[coluna]
    elif coluna.startswith(("ghi_", "gti10_")):
        prefixo = "ghi" if coluna.startswith("ghi_") else "gti10"
        tipo = (
            "Irradiação global horizontal (GHI)"
            if prefixo == "ghi"
            else "Irradiação global inclinada (GTI10, rótulo da fonte)"
        )
        fonte = "LABREN / CCST / INPE — Atlas Brasileiro de Energia Solar (2017)"
        referencia = "Série histórica 1999–2015"
        limitacao = limite_inpe
        if coluna.endswith("_longitude"):
            definicao = f"Longitude da coordenada municipal usada pelo {tipo}."
            unidade = "graus decimais"
        elif coluna.endswith("_latitude"):
            definicao = f"Latitude da coordenada municipal usada pelo {tipo}."
            unidade = "graus decimais"
        else:
            parte = coluna[len(prefixo) + 1:].replace("_Wh_m2_day", "")
            periodo = (
                "média anual"
                if parte == "annual"
                else f"média diária de {meses_pt[parte]}"
            )
            definicao = f"{tipo}: {periodo}."
            unidade = "Wh/m².dia"
            if prefixo == "gti10":
                limitacao += (
                    " Confira os metadados antes de afirmar a inclinação "
                    "associada ao rótulo GTI10."
                )
    else:
        raise ValueError(
            f"Campo sem definição documentada no dicionário: {coluna}"
        )

    registros_dicionario.append({
        "campo": coluna,
        "definição": definicao,
        "unidade": unidade,
        "fonte": fonte,
        "referência": referencia,
        "limitações": limitacao,
    })

dicionario_base_integrada = pd.DataFrame(registros_dicionario)
if set(dicionario_base_integrada["campo"]) != set(integrada_nova.columns):
    raise ValueError("O dicionário não cobre todos os campos exportados.")

dicionario_base_integrada.to_csv(
    caminho_dicionario,
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
)
dicionario_exportado = pd.read_csv(
    caminho_dicionario,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    dtype={"campo": "string"},
)
if set(dicionario_exportado["campo"].tolist()) != set(integrada_nova.columns):
    raise ValueError("O dicionário exportado não cobre todos os campos.")

resumo_valores_qualidade = (
    auditoria_valores.groupby("base", as_index=False)
    .agg(
        variaveis_numericas=("coluna", "count"),
        ausencias=("ausentes", "sum"),
        valores_nao_numericos=("não_numéricos", "sum"),
        valores_nao_finitos=("não_finitos", "sum"),
    )
)

def _formatar_valor_md(valor):
    if pd.isna(valor):
        return ""
    return str(valor).replace("|", "\\|").replace("\n", " ")


def _tabela_markdown(tabela):
    if tabela.empty:
        return "_Sem registros._"
    cabecalho = "| " + " | ".join(
        _formatar_valor_md(coluna) for coluna in tabela.columns
    ) + " |"
    separador = "| " + " | ".join("---" for _ in tabela.columns) + " |"
    linhas = [cabecalho, separador]
    for valores in tabela.itertuples(index=False, name=None):
        linhas.append(
            "| " + " | ".join(_formatar_valor_md(valor) for valor in valores)
            + " |"
        )
    return "\n".join(linhas)


data_execucao = datetime.now(
    ZoneInfo("America/Fortaleza")
).isoformat(timespec="seconds")
responsavel_execucao = "Equipe do projeto (identificação individual não informada)"
campos_pendentes_md = _tabela_markdown(status_ausencias)
granularidade_md = _tabela_markdown(auditoria_granularidade)
juncoes_md = _tabela_markdown(auditoria_juncoes)
indicadores_md = _tabela_markdown(comparacao_indicadores)
resumo_valores_md = _tabela_markdown(resumo_valores_qualidade)
auditoria_valores_md = _tabela_markdown(auditoria_valores)

relatorio_qualidade = f"""# Relatório de qualidade — base integrada Ativa Solar

- Execução: {data_execucao} (America/Fortaleza)
- Responsável: {responsavel_execucao}
- Municípios exportados: {len(integrada_nova)}
- Chave: codigo_ibge, preservado como texto.

## Fontes e referências

- INPE/LABREN, Atlas Brasileiro de Energia Solar (2017), série histórica descrita como 1999–2015; valores de irradiação em Wh/m².dia.
- ANEEL, agregados municipais de MMGD fotovoltaica com referência em 10/11/2026; contagens e potência não representam vendas da Ativa Solar.
- IBGE, estimativa populacional de 2026, referência em 01/07/2026.
- Domicílios do Censo 2022, estabelecimentos por CNAE/CNPJ e confirmação da área de atendimento continuam pendentes.

## Granularidade e junções

Cada fonte foi conferida para uma linha por código IBGE. Não houve remoção ou agregação adicional de linhas.

{granularidade_md}

Junções realizadas por codigo_ibge com validação um-para-um e indicador de correspondência:

{juncoes_md}

## Validação de valores e ausências

Valores numéricos foram verificados quanto a ausências inesperadas, conversão numérica, valores não finitos, contagens, sinais, coordenadas e meses esperados. Os extremos de irradiação foram registrados para conferência; não foram aplicados limites arbitrários nem correções sem evidência.

{resumo_valores_md}

Auditoria numérica por variável, incluindo ausências, extremos observados e situação:

{auditoria_valores_md}

Campos sem coleta ou confirmação permanecem vazios e classificados separadamente:

{campos_pendentes_md}

Nenhuma ausência foi convertida em zero. Zeros existentes nas fontes são mantidos como valores observados. Os campos atuais não foram classificados como “não disponível na fonte” ou “não se aplica”; essas categorias permanecem distintas de “não coletado” e “pendente”. Nenhum valor foi imputado ou corrigido sem confirmação na fonte.

## Indicadores recalculados

- gd_fv_kw_por_1000_hab = potência instalada em kW / população estimada × 1.000.
- gd_fv_instalacoes_por_1000_hab = instalações / população estimada × 1.000.
- Ambos foram arredondados a duas casas decimais e comparados aos valores existentes.

{indicadores_md}

Essas razões descrevem a presença municipal de geração distribuída. Não medem clientes, vendas, faturamento, crescimento anual ou potencial de mercado da empresa.

## Dicionário, unidades e limitações

O arquivo dicionario_base_integrada.csv registra definição, unidade, fonte, período de referência e limitações de cada campo exportado. A irradiação representa a referência histórica nas coordenadas municipais, não a média de todos os imóveis ou o desempenho de um telhado. O rótulo GTI10 deve ser interpretado conforme os metadados do Atlas.

## Alterações e exportação

- Fontes originais mantidas intactas.
- Nenhuma linha descartada ou somada.
- Indicadores derivados recalculados; não foram encontradas divergências após arredondamento a duas casas.
- Campos pendentes preservados como ausentes.
- Saídas: base_integrada_limpa.csv, dicionario_base_integrada.csv e este relatório.
"""

caminho_relatorio_qualidade.write_text(
    relatorio_qualidade,
    encoding="utf-8",
)

base_exportada = pd.read_csv(
    caminho_base_limpa,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    dtype={"codigo_ibge": "string"},
)
codigos_exportados = set(base_exportada["codigo_ibge"].dropna().tolist())
if len(base_exportada) != len(municipios_referencia):
    raise ValueError("A exportação não contém exatamente os 12 municípios.")
if base_exportada["codigo_ibge"].nunique() != len(municipios_referencia):
    raise ValueError("A exportação contém códigos IBGE duplicados.")
if codigos_exportados != set(municipios_referencia):
    raise ValueError("Há códigos ausentes ou extras no CSV exportado.")
if base_exportada["codigo_ibge"].dtype.name != "string":
    raise TypeError("codigo_ibge não foi reaberto como texto.")
if not base_exportada[list(campos_pendentes)].isna().all().all():
    raise ValueError("Algum campo pendente foi preenchido na exportação.")
if not pd.api.types.is_numeric_dtype(
    base_exportada["gd_fv_potencia_instalada_kw"]
):
    raise TypeError("A potência não foi reaberta como valor numérico.")
if not pd.api.types.is_numeric_dtype(
    base_exportada["gd_fv_kw_por_1000_hab"]
):
    raise TypeError("O indicador não foi reaberto como valor numérico.")

with caminho_base_limpa.open("rb") as arquivo:
    possui_bom_utf8 = arquivo.read(3) == b"\xef\xbb\xbf"
if not possui_bom_utf8:
    raise ValueError("O CSV exportado não contém BOM UTF-8.")

if len(dicionario_base_integrada) != len(integrada_nova.columns):
    raise ValueError("O dicionário não tem uma linha por campo exportado.")

print(f"Base limpa salva: {caminho_base_limpa}")
print(f"Dicionário salvo: {caminho_dicionario}")
print(f"Relatório de qualidade salvo: {caminho_relatorio_qualidade}")
print(
    f"Conferência do CSV: linhas={len(base_exportada)}, "
    f"códigos únicos={base_exportada['codigo_ibge'].nunique()}, "
    f"BOM UTF-8={possui_bom_utf8}."
)

if "files" in globals() and hasattr(files, "download"):
    caminho_zip_colab = Path("entrega_DEILAS_etapas_05_a_08.zip")
    with ZipFile(caminho_zip_colab, "w", ZIP_DEFLATED) as arquivo_zip:
        for caminho in (
            caminho_base_limpa,
            caminho_dicionario,
            caminho_relatorio_qualidade,
        ):
            arquivo_zip.write(caminho, arcname=caminho.name)
    files.download(str(caminho_zip_colab))
else:
    print("No Colab, execute a célula para baixar o ZIP com as três saídas.")